# Notebook modificado

Este notebook realiza el scraping de productos de Telcel directamente desde las páginas de listado, sin navegar a cada producto individual. Extrae la información principal de cada producto y exporta los resultados por capacidad, además de generar una base de datos unificada en Excel.

Categorías incluidas:

- **Teléfonos** y **Tablets**: se recorren por capacidad usando el filtro *Memoria* (`storage:`).
- **Accesorios** (`/tienda/categoria/accesorios`) y **Vida Conectada** (`/tienda/categoria/vida_conectada`): estas categorías **no tienen filtro de Memoria**, así que se recorren página por página hasta que ya no aparecen productos nuevos. Se les asigna `Capacidad = "NA"`.


## 1. Importar Librerías


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import re

## 2. Configurar Opciones de Selenium

Configura las opciones del navegador Chrome para realizar el scraping de manera eficiente.

In [10]:
from selenium.webdriver.chrome.options import Options

options = Options()
#options.add_argument("--headless=new")  # Modo headless para estabilidad
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-extensions")
options.add_argument("--disable-background-networking")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

# Resto de tu código para inicializar el driver y realizar el scraping
# service = Service('ruta/al/controlador/chromedriver')
# driver = webdriver.Chrome(service=service, options=options)

# Tu código para el scraping aquí

# driver.quit()

## 3. Función para Extraer Información


In [11]:
import pandas as pd
import re

# Carga de colores (si existe el archivo). Si no existe, no detiene el scrapeo.
try:
    colores_df = pd.read_csv("colores telcel.csv", dtype={"SKU": str, "Color": str})
    sku_color_dict = dict(zip(colores_df["SKU"], colores_df["Color"]))
except Exception as e:
    print(f"Aviso: no se pudo cargar 'colores telcel.csv': {e}")
    sku_color_dict = {}

def extraer_info_producto(product, capacidad, fecha_consulta, categoria):
    """
    Extrae datos desde una card del listado (telefonos o tablets).
    categoria: 'Telefonos' o 'Tablets' (se usa internamente; no se exporta)
    """
    marca = ""
    modelo = ""
    marca_modelo = product.find("p", class_="cx-product-model")
    if marca_modelo:
        partes = marca_modelo.text.split("|")
        if len(partes) == 2:
            marca = partes[0].strip()
            modelo = partes[1].strip()
        else:
            marca = marca_modelo.text.strip()

    nombre_tag = product.find("h2", class_="a-product-name")
    nombre = nombre_tag.text.strip() if nombre_tag else ""

    sku = ""
    sku_tag = product.find("span", class_="sku")
    if sku_tag:
        sku = sku_tag.text.replace("SKU:", "").strip()

    url_producto = ""
    link_tag = None
    for a_tag in product.find_all("a"):
        if a_tag.has_attr("href") and "/producto/" in a_tag["href"]:
            link_tag = a_tag
            break
    if link_tag and link_tag.has_attr("href"):
        href = link_tag["href"]
        if href.startswith("/"):
            url_producto = "https://www.telcel.com" + href
        elif href.startswith("https://"):
            url_producto = href

        # Fallback: si no viene el SKU en el card, extraerlo del final del URL.
        if not sku and url_producto:
            match = re.search(r"/(\d+)$", url_producto.rstrip("/"))
            if match:
                sku = match.group(1)

    # Validar que el URL no esté vacío, no sea None y tenga un formato válido
    if not url_producto or url_producto.strip() == "" or not url_producto.startswith("https://"):
        return None

    color = sku_color_dict.get(sku, "")

    # Precios
    precio_original = ""
    precio_descuento = ""
    porcentaje_descuento = ""

    precio_original_tag = product.find("span", class_="price-list")
    if precio_original_tag:
        precio_original = precio_original_tag.text.strip()

    price_plan_tags = product.find_all("div", class_="cx-product-price-plan")
    precios_encontrados = []
    for price_plan_tag in price_plan_tags:
        spans = price_plan_tag.find_all("span")
        for s in spans:
            txt = s.text.strip().replace('$', '').replace(',', '').replace(' ', '')
            if txt and txt.replace('.', '').isdigit():
                precios_encontrados.append(s.text.strip())

    if precio_original and precios_encontrados:
        precio_descuento = precios_encontrados[0]
    elif not precio_original and precios_encontrados:
        precio_original = precios_encontrados[0]
        precio_descuento = "No discount"
    elif precio_original and not precios_encontrados:
        precio_descuento = "No discount"
    else:
        precio_original = "No inventory"
        precio_descuento = "No discount"

    def limpiar(precio):
        return float(re.sub(r'[^\d.]', '', precio)) if precio and precio not in ["No discount", "No inventory"] else None
    p_ori = limpiar(precio_original)
    p_desc = limpiar(precio_descuento)
    if p_ori and p_desc and p_ori > p_desc:
        porcentaje_descuento = round(100 * (p_ori - p_desc) / p_ori, 2)
        porcentaje_str = f"{porcentaje_descuento}%"
    else:
        porcentaje_str = "No discount"

    return {
        "Marca": marca,
        "Modelo": modelo,
        "Nombre": nombre,
        "Color": color,
        "Capacidad": capacidad,
        "SKU": sku,
        "URL": url_producto,
        "FechaConsulta": fecha_consulta,
        "Precio_original": precio_original,
        "Precio_descuento": precio_descuento,
        "Porcentaje_descuento": porcentaje_str  
    }

## 4. Scrapear Productos por Capacidad

Teléfonos y tablets se recorren por capacidad (filtro *Memoria*). Accesorios y Vida Conectada no tienen ese filtro, así que se recorren por paginación simple.

In [12]:
def obtener_paginas_por_capacidad(url_categoria, filtro_texto_boton="Memoria"):
    """
    Detecta cuántas páginas hay por capacidad en el filtro de Memoria.
    Expande la sección solo si está cerrada y hace clic en todos los 'Mostrar más'
    hasta que no quede ninguno visible.
    """
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url_categoria)
        time.sleep(10)

        # Expandir filtro de Memoria SOLO si los links de capacidad aún no son visibles
        try:
            capacity_links = driver.find_elements("xpath", "//a[contains(@href, 'storage:')]")
            if not capacity_links:
                memoria_btn = driver.find_element("xpath", f"//button[contains(., '{filtro_texto_boton}')]")
                memoria_btn.click()
                time.sleep(3)
                print(f"Filtro '{filtro_texto_boton}' expandido.")
            else:
                print(f"Filtro '{filtro_texto_boton}' ya estaba abierto: {len(capacity_links)} capacidades visibles.")
        except Exception as e:
            print(f"No se pudo verificar/expandir el filtro '{filtro_texto_boton}': {e}")

        # Hacer clic en TODOS los botones "Mostrar más" hasta que no quede ninguno.
        # Cada clic lo convierte en "Mostrar menos", así el loop termina naturalmente.
        for i in range(20):
            try:
                btn = driver.find_element(
                    "xpath",
                    "//button[contains(normalize-space(.), 'Mostrar más')]"
                    " | //a[contains(normalize-space(.), 'Mostrar más')]"
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(3)
                print(f"  Clic en 'Mostrar más' #{i + 1}")
            except Exception:
                print(f"  Sin más botones 'Mostrar más' (total: {i} clics)")
                break

        soup = BeautifulSoup(driver.page_source, "html.parser")
        capacidades_paginas = {}
        for a_tag in soup.find_all("a", href=True):
            if "storage:" in a_tag["href"]:
                texto = a_tag.text.strip()
                match = re.search(r'(\d+\s?(MB|GB|TB|NA))', texto)
                count_tag = a_tag.find("span", class_="count")
                if match and count_tag:
                    raw = match.group(1)
                    # Normalizar "256GB" -> "256 GB" para evitar duplicados
                    norm = re.sub(r'(\d+)\s*(MB|GB|TB|NA)', r'\1 \2', raw)
                    total_items = int(count_tag.text.strip())
                    num_pages = (total_items + 15) // 16  # 16 productos por página
                    capacidades_paginas[norm] = num_pages
        return capacidades_paginas
    finally:
        try:
            driver.quit()
        except:
            pass

# URLs de categorías
URL_TELEFONOS = "https://www.telcel.com/tienda/categoria/telefonos-y-smartphones?query=:relevance:allCategories:telefonos-y-smartphones"
URL_TABLETS = "https://www.telcel.com/tienda/categoria/tablets"

# Categorías SIN filtro de Memoria (se recorren por paginación simple, Capacidad = "NA")
URL_ACCESORIOS = "https://www.telcel.com/tienda/categoria/accesorios"
URL_VIDA_CONECTADA = "https://www.telcel.com/tienda/categoria/vida_conectada"

# Lista de (url, nombre_categoria) para el scrapeo sin capacidad
CATEGORIAS_SIN_CAPACIDAD = [
    (URL_ACCESORIOS, "Accesorios"),
    (URL_VIDA_CONECTADA, "Vida Conectada"),
]

capacidades_tel = obtener_paginas_por_capacidad(URL_TELEFONOS)
print("Capacidades y páginas detectadas (Teléfonos):")
for capacidad, paginas in capacidades_tel.items():
    print(f"  - {capacidad}: {paginas} páginas")

capacidades_tab = obtener_paginas_por_capacidad(URL_TABLETS)
print("Capacidades y páginas detectadas (Tablets):")
for capacidad, paginas in capacidades_tab.items():
    print(f"  - {capacidad}: {paginas} páginas")

Filtro 'Memoria' ya estaba abierto: 6 capacidades visibles.
  Clic en 'Mostrar más' #1
  Clic en 'Mostrar más' #2
  Clic en 'Mostrar más' #3
  Sin más botones 'Mostrar más' (total: 3 clics)
Capacidades y páginas detectadas (Teléfonos):
  - 256 GB: 24 páginas
  - 512 GB: 17 páginas
  - 128 GB: 12 páginas
  - 1 TB: 3 páginas
  - 64 GB: 2 páginas
  - 2 TB: 1 páginas
  - 32 GB: 1 páginas
  - 32 MB: 1 páginas
  - 512 MB: 1 páginas
Filtro 'Memoria' ya estaba abierto: 4 capacidades visibles.
  Clic en 'Mostrar más' #1
  Sin más botones 'Mostrar más' (total: 1 clics)
Capacidades y páginas detectadas (Tablets):
  - 128 GB: 1 páginas
  - 32 GB: 1 páginas
  - 64 GB: 1 páginas
  - 256 GB: 1 páginas


In [13]:
try:
    from zoneinfo import ZoneInfo  # py3.9+
    TZ_CDMX = ZoneInfo("America/Mexico_City")
except Exception:
    TZ_CDMX = None

# Fecha/hora fija por corrida (para que toda la ejecución tenga el mismo timestamp)
FECHA_CORRIDA = (
    datetime.now(TZ_CDMX) if TZ_CDMX is not None else datetime.now()
).strftime("%Y-%m-%d %H:%M:%S")

def construir_url_listado(url_base_categoria, capacidad, page):
    capacidad_encoded = capacidad.replace(" ", "%20")

    # Teléfonos: query conocida
    if "telefonos-y-smartphones" in url_base_categoria:
        if page == 0:
            return (
                "https://www.telcel.com/tienda/categoria/telefonos-y-smartphones"
                f"?query=:relevance:allCategories:telefonos-y-smartphones:storage:{capacidad_encoded}"
            )
        return (
            "https://www.telcel.com/tienda/categoria/telefonos-y-smartphones"
            f"?query=:relevance:allCategories:telefonos-y-smartphones:storage:{capacidad_encoded}&currentPage={page}"
        )

    # Tablets: intentamos query similar; si Telcel cambia el parámetro, ajustar aquí.
    if page == 0:
        return f"{url_base_categoria}?query=:relevance:allCategories:tablets:storage:{capacidad_encoded}"
    return f"{url_base_categoria}?query=:relevance:allCategories:tablets:storage:{capacidad_encoded}&currentPage={page}"

def construir_url_listado_simple(url_base_categoria, page):
    """
    Construye la URL de una categoría SIN filtro de capacidad (accesorios, vida conectada).
    Página 0 = URL base; a partir de la 1 se agrega currentPage.
    """
    if page == 0:
        return url_base_categoria
    sep = "&" if "?" in url_base_categoria else "?"
    return f"{url_base_categoria}{sep}currentPage={page}"


def _dedupe_key(item: dict) -> str:
    # Prioridad: SKU (si existe) y luego URL
    sku = str(item.get("SKU", "") or "").strip()
    url = str(item.get("URL", "") or "").strip()
    return sku if sku else url

def scrape_capacidad(url_categoria, categoria_nombre, capacidad, num_pages):
    print(f"Scrapeando {categoria_nombre} - capacidad: {capacidad}")
    data = []
    vistos = set()  # Evita repetidos (por SKU o URL) dentro de la capacidad
    for page in range(num_pages):
        url = construir_url_listado(url_categoria, capacidad, page)

        driver = None
        for intento in range(3):  # Hasta 3 intentos por página
            try:
                driver = webdriver.Chrome(options=options)
                driver.get(url)
                time.sleep(12)
                soup = BeautifulSoup(driver.page_source, "html.parser")
                products = soup.find_all("cx-custom-product-grid-item", class_="cx-grid-item")
                if not products:
                    products = soup.find_all("div", class_="product-card")

                # Usar la misma FechaConsulta para toda la corrida
                fecha_consulta = FECHA_CORRIDA
                for product in products:
                    info = extraer_info_producto(product, capacidad, fecha_consulta, categoria_nombre)
                    if info is None:
                        continue
                    k = _dedupe_key(info)
                    if not k:
                        continue
                    if k in vistos:
                        continue
                    vistos.add(k)
                    data.append(info)
                break  # Éxito
            except Exception as e:
                print(f"  Error en {categoria_nombre} {capacidad} página {page}, intento {intento+1}: {e}")
                time.sleep(5)
            finally:
                if driver:
                    try:
                        driver.quit()
                    except:
                        pass

    print(f"Total productos {categoria_nombre} {capacidad}: {len(data)} (sin duplicados, con URL válido)")
    return (categoria_nombre, capacidad), data

def scrape_categoria_sin_capacidad(url_categoria, categoria_nombre, max_paginas=60):
    """
    Scrapea una categoría que NO tiene filtro de Memoria (accesorios, vida conectada).
    Recorre páginas con currentPage hasta que una página no aporte productos nuevos
    (o hasta max_paginas como tope de seguridad).
    """
    print(f"Scrapeando {categoria_nombre} (sin filtro de capacidad)")
    capacidad = "NA"
    data = []
    vistos = set()

    for page in range(max_paginas):
        url = construir_url_listado_simple(url_categoria, page)
        nuevos_en_pagina = 0
        pagina_ok = False

        driver = None
        for intento in range(3):  # Hasta 3 intentos por página
            try:
                driver = webdriver.Chrome(options=options)
                driver.get(url)
                time.sleep(12)
                soup = BeautifulSoup(driver.page_source, "html.parser")
                products = soup.find_all("cx-custom-product-grid-item", class_="cx-grid-item")
                if not products:
                    products = soup.find_all("div", class_="product-card")

                fecha_consulta = FECHA_CORRIDA
                for product in products:
                    info = extraer_info_producto(product, capacidad, fecha_consulta, categoria_nombre)
                    if info is None:
                        continue
                    k = _dedupe_key(info)
                    if not k or k in vistos:
                        continue
                    vistos.add(k)
                    data.append(info)
                    nuevos_en_pagina += 1

                pagina_ok = True
                break  # Éxito
            except Exception as e:
                print(f"  Error en {categoria_nombre} página {page}, intento {intento+1}: {e}")
                time.sleep(5)
            finally:
                if driver:
                    try:
                        driver.quit()
                    except:
                        pass

        # Si la página no trajo productos nuevos, asumimos que ya no hay más resultados
        if not pagina_ok or nuevos_en_pagina == 0:
            print(f"  {categoria_nombre}: fin de la paginación en página {page}")
            break

    print(f"Total productos {categoria_nombre} {capacidad}: {len(data)} (sin duplicados, con URL válido)")
    return (categoria_nombre, capacidad), data


resultados = {}

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = []
    for capacidad, num_pages in capacidades_tel.items():
        futures.append(executor.submit(scrape_capacidad, URL_TELEFONOS, "Telefonos", capacidad, num_pages))
    for capacidad, num_pages in capacidades_tab.items():
        futures.append(executor.submit(scrape_capacidad, URL_TABLETS, "Tablets", capacidad, num_pages))

    # Categorías sin filtro de capacidad (accesorios, vida conectada)
    for url_cat, nombre_cat in CATEGORIAS_SIN_CAPACIDAD:
        futures.append(executor.submit(scrape_categoria_sin_capacidad, url_cat, nombre_cat))

    for future in as_completed(futures):
        key, data = future.result()
        resultados[key] = data

Scrapeando Telefonos - capacidad: 256 GB
Scrapeando Telefonos - capacidad: 512 GB
Scrapeando Telefonos - capacidad: 128 GB
Scrapeando Telefonos - capacidad: 1 TB
Total productos Telefonos 1 TB: 36 (sin duplicados, con URL válido)
Scrapeando Telefonos - capacidad: 64 GB
Total productos Telefonos 64 GB: 29 (sin duplicados, con URL válido)
Scrapeando Telefonos - capacidad: 2 TB
Total productos Telefonos 2 TB: 3 (sin duplicados, con URL válido)
Scrapeando Telefonos - capacidad: 32 GB
Total productos Telefonos 32 GB: 3 (sin duplicados, con URL válido)
Scrapeando Telefonos - capacidad: 32 MB
Total productos Telefonos 32 MB: 2 (sin duplicados, con URL válido)
Scrapeando Telefonos - capacidad: 512 MB
Total productos Telefonos 512 MB: 2 (sin duplicados, con URL válido)
Scrapeando Tablets - capacidad: 128 GB
Total productos Tablets 128 GB: 14 (sin duplicados, con URL válido)
Scrapeando Tablets - capacidad: 32 GB
Total productos Tablets 32 GB: 3 (sin duplicados, con URL válido)
Scrapeando Tablets

## 5. Exportar Resultados de Cada Capacidad a CSV

Convierte los datos extraídos en DataFrames y expórtalos a archivos CSV por cada capacidad.

In [14]:
import re

for (categoria, capacidad), data in resultados.items():
    if data:
        # Nombre de archivo seguro (quita acentos/espacios: "Vida Conectada" -> "vidaconectada")
        safe_cat = re.sub(r"[^a-z0-9]", "", categoria.lower())
        safe_cap = re.sub(r"[^a-z0-9]", "", capacidad.lower())
        nombre_archivo = f"telcel_{safe_cat}_{safe_cap}.csv"
        df = pd.DataFrame(data)
        df.to_csv(nombre_archivo, index=False, encoding="utf-8")
        print(f"Datos exportados a {nombre_archivo}")
    else:
        print(f"No se encontraron productos para {categoria} - {capacidad}.")


Datos exportados a telcel_telefonos_1tb.csv
Datos exportados a telcel_telefonos_64gb.csv
Datos exportados a telcel_telefonos_2tb.csv
Datos exportados a telcel_telefonos_32gb.csv
Datos exportados a telcel_telefonos_32mb.csv
Datos exportados a telcel_telefonos_512mb.csv
Datos exportados a telcel_tablets_128gb.csv
Datos exportados a telcel_tablets_32gb.csv
Datos exportados a telcel_tablets_64gb.csv
Datos exportados a telcel_telefonos_128gb.csv
Datos exportados a telcel_tablets_256gb.csv
Datos exportados a telcel_telefonos_512gb.csv
Datos exportados a telcel_vidaconectada_na.csv
Datos exportados a telcel_telefonos_256gb.csv
Datos exportados a telcel_accesorios_na.csv


## 6. Unir Bases de Datos y Exportar a Excel

Lee los archivos CSV generados, concatena los DataFrames, ordena por capacidad y exporta el resultado final a un archivo Excel.

In [ ]:
import pandas as pd
from datetime import datetime
import glob

try:
    from zoneinfo import ZoneInfo  # py3.9+
except Exception:
    ZoneInfo = None

# Toma todos los CSV generados por este notebook (telefonos y tablets)
archivos = sorted(glob.glob("telcel_*.csv"))

dfs = []
for archivo in archivos:
    try:
        df = pd.read_csv(archivo, encoding="utf-8", keep_default_na=False, na_values=[])
        # Filtros más robustos para URLs vacíos o inválidos
        if "URL" in df.columns:
            df = df[df["URL"].notna()]
            df = df[df["URL"].astype(str).str.strip() != ""]
            df = df[df["URL"].astype(str).str.startswith("https://")]
        if not df.empty:
            dfs.append(df)
            print(f"Archivo {archivo}: {len(df)} productos válidos con URL")
        else:
            print(f"Archivo {archivo}: No se encontraron productos con URL válido")
    except Exception as e:
        print(f"No se pudo leer {archivo}: {e}")

if dfs:
    df_final = pd.concat(dfs, ignore_index=True)

    # Deduplicación global (por SKU si existe; si no, por URL)
    if "SKU" in df_final.columns:
        sku_nonempty = df_final["SKU"].astype(str).str.strip().ne("")
        df_sku = df_final[sku_nonempty].drop_duplicates(subset=["SKU"], keep="first")
        df_no_sku = df_final[~sku_nonempty]
        if "URL" in df_no_sku.columns:
            df_no_sku = df_no_sku.drop_duplicates(subset=["URL"], keep="first")
        df_final = pd.concat([df_sku, df_no_sku], ignore_index=True)
    elif "URL" in df_final.columns:
        df_final = df_final.drop_duplicates(subset=["URL"], keep="first")

    # Eliminar columna Categoria (si existe)
    if "Categoria" in df_final.columns:
        df_final = df_final.drop(columns=["Categoria"])

    # FechaConsulta: asegurar un valor consistente sin recortar el dataset.
    # Nota: NO filtramos por max FechaConsulta porque eso puede eliminar categorías/capacidades completas.
    if "FechaConsulta" in df_final.columns:
        fc = pd.to_datetime(df_final["FechaConsulta"], errors="coerce")
        max_fc = fc.max()
        if pd.notna(max_fc):
            df_final["FechaConsulta"] = max_fc.strftime("%Y-%m-%d %H:%M:%S")

    # Fecha para el nombre del archivo (hora local de México)
    if ZoneInfo is not None:
        ahora = datetime.now(ZoneInfo("America/Mexico_City"))
    else:
        ahora = datetime.now()
    fecha_hoy = ahora.strftime("%Y-%m-%d")
    hora_hoy = ahora.strftime("%H%M%S")

    # Incluir hora para no sobrescribir/abrir versiones viejas
    nombre_archivo_excel = f"Daily Pricing Mapping Telcel Cacs ({fecha_hoy}) {hora_hoy} V3.xlsx"
    df_final.to_excel(nombre_archivo_excel, index=False)
    print(f"Archivo combinado guardado como {nombre_archivo_excel}")
    print(f"Total de productos con URL válido (sin duplicados): {len(df_final)}")
else:
    print("No se encontraron archivos válidos para combinar.")

Archivo telcel_accesorios_na.csv: 177 productos válidos con URL
Archivo telcel_tablets_128gb.csv: 14 productos válidos con URL
Archivo telcel_tablets_256gb.csv: 1 productos válidos con URL
Archivo telcel_tablets_32gb.csv: 3 productos válidos con URL
Archivo telcel_tablets_64gb.csv: 3 productos válidos con URL
Archivo telcel_telefonos_128gb.csv: 190 productos válidos con URL
Archivo telcel_telefonos_1tb.csv: 36 productos válidos con URL
Archivo telcel_telefonos_256gb.csv: 357 productos válidos con URL
Archivo telcel_telefonos_2tb.csv: 3 productos válidos con URL
Archivo telcel_telefonos_32gb.csv: 3 productos válidos con URL
Archivo telcel_telefonos_32mb.csv: 2 productos válidos con URL
Archivo telcel_telefonos_512gb.csv: 262 productos válidos con URL
Archivo telcel_telefonos_512mb.csv: 2 productos válidos con URL
Archivo telcel_telefonos_64gb.csv: 29 productos válidos con URL
Archivo telcel_vidaconectada_na.csv: 133 productos válidos con URL
Archivo combinado guardado como Daily Pricing

In [16]:
# Diagnóstico de zona horaria y conversión de FechaConsulta
from datetime import datetime
try:
    from zoneinfo import ZoneInfo
except Exception:
    ZoneInfo = None

print("--- Diagnóstico ---")
print("Hora local del sistema:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Hora local con tzinfo:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z (%Z)"))
if ZoneInfo is not None:
    print("Hora CDMX:", datetime.now(ZoneInfo("America/Mexico_City")).strftime("%Y-%m-%d %H:%M:%S %z (%Z)"))
else:
    print("ZoneInfo no disponible en este entorno")

# Ejemplo de conversión usando la fecha que reportaste
ejemplo = "2026-04-24 09:07:11"
parsed = pd.to_datetime([ejemplo], errors="coerce")
if ZoneInfo is not None:
    local_tz = datetime.now().astimezone().tzinfo
    parsed_local = parsed.tz_localize(local_tz)
    parsed_cdmx = parsed_local.tz_convert(ZoneInfo("America/Mexico_City"))
    print("Ejemplo interpretado como local -> CDMX:", parsed_cdmx.strftime("%Y-%m-%d %H:%M:%S %z (%Z)")[0])
else:
    print("Ejemplo (sin tz):", parsed[0])

--- Diagnóstico ---
Hora local del sistema: 2026-09-01 09:15:27
Hora local con tzinfo: 2026-09-01 09:15:27 -0600 (Central Standard Time (Mexico))
Hora CDMX: 2026-09-01 09:15:27 -0600 (CST)
Ejemplo interpretado como local -> CDMX: 2026-04-24 09:07:11 -0600 (CST)
